# Data Class Smell Detection
## Consolidated Notebook: Rules → Traditional ML → TabPFN → Ensemble
### GroupKFold cross-validation | All methods | Deployment-ready

| Input | Path |
|---|---|
| Industrial | `drive/MyDrive/codesmell/data/smellycodefull.csv` |
| Student    | `drive/MyDrive/codesmell/data/student_features_with_labels.csv` |
| Output     | `drive/MyDrive/codesmellimprovements/god_class/` |

In [ ]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
!pip install xgboost imbalanced-learn scikit-learn joblib pandas numpy -q
!pip install tabpfn -q
!pip install lightgbm -q
print("All packages installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.8/229.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
All packages installed.


In [ ]:
# ── Cell 2: Imports + Drive mount + Global storage ───────────────────────────
import os, json, warnings, joblib, pickle
import numpy  as np
import pandas as pd
import xgboost as xgb
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Set TabPFN token before any tabpfn import
os.environ["TABPFN_TOKEN"] = ""
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from sklearn.ensemble          import RandomForestClassifier
from sklearn.tree              import DecisionTreeClassifier
from sklearn.linear_model      import LogisticRegression
from sklearn.preprocessing     import StandardScaler
from sklearn.model_selection   import GroupKFold, StratifiedKFold
from sklearn.metrics           import (
    f1_score, precision_score, recall_score,
    average_precision_score, confusion_matrix,
    precision_recall_curve
)
from imblearn.combine          import SMOTETomek, SMOTEENN
from imblearn.over_sampling    import SMOTE
from imblearn.pipeline         import Pipeline as ImbPipeline

# ── Global OOF accumulator ────────────────────────────────────────────────────
# Each method cell populates:
#   ALL_RESULTS[model_name] = {'prob': np.array, 'pred_05': np.array, 'category': str}
ALL_RESULTS = {}

print(f"XGBoost {xgb.__version__}  ready")
print("ALL_RESULTS dict initialised — ready to accumulate OOF predictions.")

Mounted at /content/drive
XGBoost 3.2.0  ready
ALL_RESULTS dict initialised — ready to accumulate OOF predictions.


In [ ]:
# ── Cell 3: Paths, constants, feature columns ────────────────────────────────
BASE      = '/content/drive/MyDrive/codesmell'
IND_DATA  = f'{BASE}/data/smellycodefull.csv'
STUD_DATA = f'{BASE}/data/student_features_with_labels.csv'

OUT_BASE   = '/content/drive/MyDrive/CodeSmell_Stage3/data_class'
P1_MODELS  = f'{OUT_BASE}/phase1/models'
P1_RESULTS = f'{OUT_BASE}/phase1/results'
PA_MODELS  = f'{OUT_BASE}/modelA/models'
PA_RESULTS = f'{OUT_BASE}/modelA/results'
PT_MODELS  = f'{OUT_BASE}/trAda/models'
PT_RESULTS = f'{OUT_BASE}/trAda/results'
DEPLOY_DIR = f'{OUT_BASE}/deployment'

for d in [P1_MODELS, P1_RESULTS, PA_MODELS, PA_RESULTS,
          PT_MODELS, PT_RESULTS, DEPLOY_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

FEATURE_COLS = [
    'Logical Lines', 'Distinct Operators', 'Distinct Operands',
    'Total Operators', 'Total Operands', 'Cyclomatic Complexity',
]
TARGET      = 'label_data_class'
IND_LABEL   = 'Data class'
IND_PROJECT = 'Project'

print("Paths configured.")
print(f"  Industrial data : {IND_DATA}")
print(f"  Student data    : {STUD_DATA}")
print(f"  Output base     : {OUT_BASE}")

Paths configured.
  Industrial data : /content/drive/MyDrive/codesmell/data/smellycodefull.csv
  Student data    : /content/drive/MyDrive/codesmell/data/student_features_with_labels.csv
  Output base     : /content/drive/MyDrive/CodeSmell_Stage3/data_class


In [ ]:
# df_ind.columns

In [ ]:
# df_ind.columns

In [ ]:
# ── Cell 4: Load industrial + student data ───────────────────────────────────
# ── Industrial ────────────────────────────────────────────────────────────────
df_ind = pd.read_csv(IND_DATA)
missing_ind = [c for c in FEATURE_COLS + [IND_LABEL, IND_PROJECT] if c not in df_ind.columns]
if missing_ind:
    raise ValueError(f"Industrial data missing columns: {missing_ind}")

df_ind[FEATURE_COLS] = df_ind[FEATURE_COLS].fillna(0)
X_ind = df_ind[FEATURE_COLS].values
y_ind = df_ind[IND_LABEL].values.astype(int)
g_ind = df_ind[IND_PROJECT].values
pos_i = int(y_ind.sum())
print(f"Industrial : {df_ind.shape[0]:,} samples | {pos_i} Data Class ({pos_i/len(y_ind)*100:.1f}%)")
print(f"  Projects : {df_ind[IND_PROJECT].nunique()}")

# ── Student ────────────────────────────────────────────────────────────────────
df_stu = pd.read_csv(STUD_DATA)
missing_stu = [c for c in FEATURE_COLS + [TARGET, 'project'] if c not in df_stu.columns]
if missing_stu:
    raise ValueError(f"Student data missing columns: {missing_stu}")

df_stu = df_stu.dropna(subset=FEATURE_COLS + [TARGET]).reset_index(drop=True)
df_stu[TARGET] = df_stu[TARGET].astype(int)

X_all      = df_stu[FEATURE_COLS].values.astype(float)
y_all      = df_stu[TARGET].values.astype(int)
groups_all = df_stu['project'].values
n_all      = len(df_stu)
n_pos_all  = int(y_all.sum())
n_repos    = df_stu['project'].nunique()

print(f"\nStudent    : {n_all:,} samples | {n_pos_all} Data Class ({n_pos_all/n_all*100:.1f}%)")
print(f"  Projects : {n_repos}")

# ── Per-repo summary ──────────────────────────────────────────────────────────
print("\nPer-repo positives:")
for repo, grp in df_stu.groupby('project'):
    p = int(grp[TARGET].sum())
    print(f"  {repo:45s}  {p:2d}  {'█'*p}")

# ── GroupKFold setup (shared by all subsequent cells) ─────────────────────────
N_SPLITS = min(5, n_repos)
while N_SPLITS > 2 and n_pos_all / N_SPLITS < 1.5:
    N_SPLITS -= 1
gkf = GroupKFold(n_splits=N_SPLITS)
print(f"\nGroupKFold: n_splits={N_SPLITS}  total_pos={n_pos_all}  ~pos/fold={n_pos_all/N_SPLITS:.1f}")

Industrial : 107,554 samples | 3284 Data Class (3.1%)
  Projects : 89

Student    : 3,266 samples | 249 Data Class (7.6%)
  Projects : 821

Per-repo positives:
  20+Java-Question-lab-work                       0  
  4sum.java                                       0  
  A.java                                          0  
  AES.java                                        0  
  AI Riddle Game                                  0  
  API-Lesson1                                     2  ██
  ATM                                             0  
  ATM System                                      5  █████
  ATM interface                                   0  
  ATM-Bankomat                                    1  █
  ATM2-Bankomat                                   1  █
  AbstractClassesAndMethods                       0  
  Abstraction & Interfaces.java                   0  
  AccessModifiers                                 1  █
  Addhar entry                                    0  
  Address Book      

In [ ]:
# ── Cell 5: Model_I — Train on industrial data, save .joblib, extract booster ─
# Used as: (a) zero-shot transfer baseline on student data
#          (b) starting checkpoint for TrAdaBoost (Cell 8)
print("="*72)
print("MODEL_I: Industrial data  |  GroupKFold/StratifiedKFold + SMOTETomek")
print("="*72)

n_proj = df_ind[IND_PROJECT].nunique()
if n_proj >= 5:
    cv_ind     = GroupKFold(n_splits=min(5, n_proj))
    split_args = (X_ind, y_ind, g_ind)
    cv_desc    = f"GroupKFold(n={min(5,n_proj)})"
else:
    cv_ind     = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    split_args = (X_ind, y_ind)
    cv_desc    = "StratifiedKFold(n=5)"

def make_ind_pipe(clf):
    return ImbPipeline([
        ('scaler',   StandardScaler()),
        ('resample', SMOTETomek(random_state=RANDOM_STATE)),
        ('clf',      clf),
    ])

pipelines_I = {
    'LogisticRegression': make_ind_pipe(
        LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)),
    'DecisionTree':       make_ind_pipe(
        DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=RANDOM_STATE)),
    'RandomForest':       make_ind_pipe(
        RandomForestClassifier(n_estimators=100, class_weight='balanced',
                               random_state=RANDOM_STATE, n_jobs=-1)),
    'XGBoost':            make_ind_pipe(
        xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                          eval_metric='logloss', random_state=RANDOM_STATE,
                          n_jobs=-1, verbosity=0)),
}

print(f"CV: {cv_desc}")
print(f"\n{'Model':22s}  {'F1':>6}  {'Prec':>6}  {'Rec':>6}  {'PR-AUC':>7}")
print("-"*56)

fitted_I = {}
for name, pipe in pipelines_I.items():
    yp  = np.zeros(len(y_ind), dtype=int)
    prb = np.zeros(len(y_ind))
    for tr, va in cv_ind.split(*split_args):
        Xtr, Xva, ytr = X_ind[tr], X_ind[va], y_ind[tr]
        if ytr.sum() < 2:
            continue
        pipe.fit(Xtr, ytr)
        yp[va]  = pipe.predict(Xva)
        prb[va] = pipe.predict_proba(Xva)[:, 1]
    pipe.fit(X_ind, y_ind)
    fitted_I[name] = pipe
    f1   = f1_score(y_ind, yp, zero_division=0)
    prec = precision_score(y_ind, yp, zero_division=0)
    rec  = recall_score(y_ind, yp, zero_division=0)
    pr   = average_precision_score(y_ind, prb)
    print(f"  {name:20s}  {f1:6.3f}  {prec:6.3f}  {rec:6.3f}  {pr:7.3f}")
    joblib.dump(pipe, f"{P1_MODELS}/model_I_data_class_{name}.joblib")

pd.DataFrame([{
    'model': n, 'f1': round(f1_score(y_ind,np.zeros(len(y_ind),dtype=int)),3)
} for n in fitted_I]).to_csv(f"{P1_RESULTS}/phase1_results.csv", index=False)

# ── Extract XGBoost booster + industrial scaler (needed for TrAda in Cell 8) ─
model_I_xgb = fitted_I['XGBoost']
booster_I   = model_I_xgb.named_steps['clf'].get_booster()
ind_scaler  = model_I_xgb.named_steps['scaler']
joblib.dump(ind_scaler, f"{P1_MODELS}/ind_scaler.joblib")

print(f"\nModel_I saved → {P1_MODELS}")
print("booster_I + ind_scaler ready (used by TrAdaBoost in Cell 8)")

MODEL_I: Industrial data  |  GroupKFold/StratifiedKFold + SMOTETomek
CV: GroupKFold(n=5)

Model                       F1    Prec     Rec   PR-AUC
--------------------------------------------------------
  LogisticRegression     0.093   0.051   0.558    0.050
  DecisionTree           0.134   0.072   0.909    0.125
  RandomForest           0.235   0.205   0.274    0.176
  XGBoost                0.218   0.130   0.664    0.198

Model_I saved → /content/drive/MyDrive/CodeSmell_Stage3/data_class/phase1/models
booster_I + ind_scaler ready (used by TrAdaBoost in Cell 8)


In [ ]:
# ── Cell 6: Rule-based baselines — GroupKFold OOF on student data ────────────
# Rules derived from feature importance analysis (Logical Lines dominates).
# Rule1  — Single:  Logical Lines > P75
# Rule2  — AND:     LL > P75 AND Total Operands > P75 AND Distinct Operands > P75
# Rule3  — OR:      any feature > P90
# Rule4  — Scoring: LL>P75 (+3), TotalOp>P75 (+2), DistOp>P75 (+1),
#                   TotalOpR>P75 (+1), CC>P75 (+1) → predict if score >= 3
print("="*72)
print("RULE-BASED BASELINES")
print("="*72)

FEAT_IDX = {f: i for i, f in enumerate(FEATURE_COLS)}
ll_idx   = FEAT_IDX['Logical Lines']
do_idx   = FEAT_IDX['Distinct Operands']
toi_idx  = FEAT_IDX['Total Operators']
top_idx  = FEAT_IDX['Total Operands']
cc_idx   = FEAT_IDX['Cyclomatic Complexity']

rule_preds = {
    'Rule1_Single':  np.zeros(n_all, dtype=int),
    'Rule2_AND':     np.zeros(n_all, dtype=int),
    'Rule3_OR':      np.zeros(n_all, dtype=int),
    'Rule4_Score':   np.zeros(n_all, dtype=int),
}

for _, (tr_idx, va_idx) in enumerate(gkf.split(X_all, y_all, groups_all)):
    X_tr, X_va = X_all[tr_idx], X_all[va_idx]
    p75 = np.percentile(X_tr, 75, axis=0)
    p90 = np.percentile(X_tr, 90, axis=0)

    rule_preds['Rule1_Single'][va_idx] = (
        X_va[:, ll_idx] > p75[ll_idx]).astype(int)

    rule_preds['Rule2_AND'][va_idx] = (
        (X_va[:, ll_idx]  > p75[ll_idx]) &
        (X_va[:, top_idx] > p75[top_idx]) &
        (X_va[:, do_idx]  > p75[do_idx])
    ).astype(int)

    rule_preds['Rule3_OR'][va_idx] = np.any(
        X_va > p90[np.newaxis, :], axis=1).astype(int)

    score = (
        (X_va[:, ll_idx]  > p75[ll_idx]).astype(int)  * 3 +
        (X_va[:, top_idx] > p75[top_idx]).astype(int) * 2 +
        (X_va[:, do_idx]  > p75[do_idx]).astype(int)  * 1 +
        (X_va[:, toi_idx] > p75[toi_idx]).astype(int) * 1 +
        (X_va[:, cc_idx]  > p75[cc_idx]).astype(int)  * 1
    )
    rule_preds['Rule4_Score'][va_idx] = (score >= 3).astype(int)

print(f"\n{'Rule':20s}  {'F1':>6}  {'Prec':>6}  {'Rec':>6}  {'PR-AUC':>7}  {'TP':>3}  {'FP':>3}")
print("-"*62)

for rname, rpred in rule_preds.items():
    f1   = f1_score(y_all, rpred, zero_division=0)
    prec = precision_score(y_all, rpred, zero_division=0)
    rec  = recall_score(y_all, rpred, zero_division=0)
    pr   = average_precision_score(y_all, rpred.astype(float))
    cm   = confusion_matrix(y_all, rpred)
    tp, fp = cm[1,1], cm[0,1]
    print(f"  {rname:18s}  {f1:6.3f}  {prec:6.3f}  {rec:6.3f}  {pr:7.3f}  {tp:3d}  {fp:3d}")
    # Rules have no continuous probability; use binary pred as proxy
    ALL_RESULTS[rname] = {
        'prob':     rpred.astype(float),
        'pred_05':  rpred,
        'category': 'Rule'
    }

print(f"\nRule baselines → ALL_RESULTS ({len(ALL_RESULTS)} entries)")

RULE-BASED BASELINES

Rule                      F1    Prec     Rec   PR-AUC   TP   FP
--------------------------------------------------------------
  Rule1_Single         0.107   0.070   0.229    0.075   57  755
  Rule2_AND            0.017   0.013   0.028    0.074    7  545
  Rule3_OR             0.012   0.009   0.020    0.075    5  568
  Rule4_Score          0.097   0.062   0.229    0.073   57  869

Rule baselines → ALL_RESULTS (4 entries)


In [ ]:
# ── Cell 7: Traditional ML — Model_A (student) + Model_I zero-shot ────────────
# Model_I  → pre-trained on industrial, applied as-is to student (zero-shot)
# Model_A  → trained from scratch on student train folds (LR / DT / RF / XGB)
print("="*72)
print("TRADITIONAL ML: Model_I (zero-shot) + Model_A (student GroupKFold OOF)")
print("="*72)

oof_mi = {m: {'prob': np.zeros(n_all), 'pred': np.zeros(n_all, dtype=int)}
          for m in ['LogisticRegression','DecisionTree','RandomForest','XGBoost']}
oof_ma = {m: {'prob': np.zeros(n_all), 'pred': np.zeros(n_all, dtype=int)}
          for m in ['LogisticRegression','DecisionTree','RandomForest','XGBoost']}

xgb_p = {
    'objective': 'binary:logistic', 'eval_metric': 'aucpr',
    'learning_rate': 0.05, 'max_depth': 4,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'seed': RANDOM_STATE, 'verbosity': 0,
}

print(f"\n{'Fold':>4}  {'TrPos':>5}  {'TePos':>5}  {'RF_F1':>7}  {'XGB_F1':>7}")
print("-"*40)

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_all, y_all, groups_all), 1):
    X_tr, X_va = X_all[tr_idx], X_all[va_idx]
    y_tr        = y_all[tr_idx]
    tr_pos      = int(y_tr.sum())
    va_pos      = int(y_all[va_idx].sum())
    neg_tr      = int((y_tr == 0).sum())
    spw         = round(neg_tr / max(tr_pos, 1), 2)

    # ── Model_I: zero-shot on student validation ───────────────────
    for mn in oof_mi:
        pipe = joblib.load(f"{P1_MODELS}/model_I_data_class_{mn}.joblib")
        oof_mi[mn]['prob'][va_idx]  = pipe.predict_proba(X_va)[:, 1]
        oof_mi[mn]['pred'][va_idx]  = (oof_mi[mn]['prob'][va_idx] >= 0.5).astype(int)

    # ── Model_A: train on student fold ────────────────────────────
    if tr_pos < 2:
        for mn in oof_ma:
            oof_ma[mn]['prob'][va_idx] = oof_mi['XGBoost']['prob'][va_idx]
            oof_ma[mn]['pred'][va_idx] = oof_mi['XGBoost']['pred'][va_idx]
        print(f"{fold:>4}  {tr_pos:>5}  {va_pos:>5}  {'fallbk':>7}  {'fallbk':>7}")
        continue

    sc   = StandardScaler().fit(X_tr)
    Xts  = sc.transform(X_tr)
    Xvs  = sc.transform(X_va)

    for mn, clf in [
        ('LogisticRegression', LogisticRegression(
            class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)),
        ('DecisionTree', DecisionTreeClassifier(
            max_depth=5, class_weight='balanced', random_state=RANDOM_STATE)),
        ('RandomForest', RandomForestClassifier(
            n_estimators=300, min_samples_leaf=1, class_weight='balanced',
            random_state=RANDOM_STATE, n_jobs=-1)),
    ]:
        clf.fit(Xts, y_tr)
        oof_ma[mn]['prob'][va_idx] = clf.predict_proba(Xvs)[:, 1]
        oof_ma[mn]['pred'][va_idx] = (oof_ma[mn]['prob'][va_idx] >= 0.5).astype(int)

    dtr = xgb.DMatrix(Xts, label=y_tr, feature_names=FEATURE_COLS)
    dva = xgb.DMatrix(Xvs,             feature_names=FEATURE_COLS)
    b   = xgb.train({**xgb_p, 'scale_pos_weight': spw}, dtr,
                    num_boost_round=100, verbose_eval=False)
    oof_ma['XGBoost']['prob'][va_idx] = b.predict(dva)
    oof_ma['XGBoost']['pred'][va_idx] = (oof_ma['XGBoost']['prob'][va_idx] >= 0.5).astype(int)

    rf_f1  = f1_score(y_all[va_idx], oof_ma['RandomForest']['pred'][va_idx], zero_division=0)
    xgb_f1 = f1_score(y_all[va_idx], oof_ma['XGBoost']['pred'][va_idx],      zero_division=0)
    print(f"{fold:>4}  {tr_pos:>5}  {va_pos:>5}  {rf_f1:>7.3f}  {xgb_f1:>7.3f}")

# ── Populate ALL_RESULTS ──────────────────────────────────────────────────────
for mn in oof_ma:
    ALL_RESULTS[f'Model_I_{mn}'] = {
        'prob': oof_mi[mn]['prob'], 'pred_05': oof_mi[mn]['pred'], 'category': 'Model_I'}
    ALL_RESULTS[f'Model_A_{mn}'] = {
        'prob': oof_ma[mn]['prob'], 'pred_05': oof_ma[mn]['pred'], 'category': 'Model_A'}

print(f"\nTraditional ML → ALL_RESULTS ({len(ALL_RESULTS)} entries)")
print(f"\n{'Model':30s}  {'PR-AUC':>7}  {'F1@0.5':>7}")
print("-"*48)
for mn in ['LogisticRegression','DecisionTree','RandomForest','XGBoost']:
    pr_a = average_precision_score(y_all, oof_ma[mn]['prob'])
    f1_a = f1_score(y_all, oof_ma[mn]['pred'], zero_division=0)
    print(f"  Model_A {mn:20s}  {pr_a:7.3f}  {f1_a:7.3f}")

TRADITIONAL ML: Model_I (zero-shot) + Model_A (student GroupKFold OOF)

Fold  TrPos  TePos    RF_F1   XGB_F1
----------------------------------------
   1    222     27    0.667    0.529
   2    191     58    0.617    0.584
   3    226     23    0.500    0.324
   4    188     61    0.685    0.580
   5    169     80    0.763    0.791

Traditional ML → ALL_RESULTS (12 entries)

Model                            PR-AUC   F1@0.5
------------------------------------------------
  Model_A LogisticRegression      0.470    0.445
  Model_A DecisionTree            0.442    0.480
  Model_A RandomForest            0.687    0.668
  Model_A XGBoost                 0.647    0.588


In [ ]:
# ── Cell 7b: RF + SMOTE variants ──────────────────────────────────────────────
# Finds best RF config (plain vs SMOTE) to feed into ensemble (Cell 11)
print("="*72)
print("RF + SMOTE variants  |  GroupKFold OOF")
print("="*72)

RF_SMOTE_STRATS = [0.1, 0.3, 0.5, 'auto']

for strat in RF_SMOTE_STRATS:
    oof_prob = np.zeros(n_all)
    oof_pred = np.zeros(n_all, dtype=int)

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_all, y_all, groups_all), 1):
        X_tr, X_va = X_all[tr_idx], X_all[va_idx]
        y_tr        = y_all[tr_idx]
        tr_pos      = int(y_tr.sum())
        tr_neg      = int((y_tr == 0).sum())
        if tr_pos < 2:
            continue

        sc  = StandardScaler().fit(X_tr)
        Xts = sc.transform(X_tr)
        Xvs = sc.transform(X_va)

        # safe ratio clamp
        if isinstance(strat, float):
            current_ratio = tr_pos / max(tr_neg, 1)
            safe_strat    = max(strat, current_ratio + 1e-6)
        else:
            safe_strat = strat

        k = min(3, tr_pos - 1) if tr_pos > 2 else 1
        X_res, y_res = SMOTE(
            sampling_strategy=safe_strat,
            k_neighbors=k,
            random_state=RANDOM_STATE
        ).fit_resample(Xts, y_tr)

        # NO class_weight — SMOTE is doing the balancing
        rf = RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=1,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
        rf.fit(X_res, y_res)
        oof_prob[va_idx] = rf.predict_proba(Xvs)[:, 1]
        oof_pred[va_idx] = (oof_prob[va_idx] >= 0.5).astype(int)

    pr   = average_precision_score(y_all, oof_prob)
    f1   = f1_score(y_all, oof_pred, zero_division=0)
    prec = precision_score(y_all, oof_pred, zero_division=0)
    rec  = recall_score(y_all, oof_pred, zero_division=0)

    label = f'RF_SMOTE_{strat}'
    ALL_RESULTS[label] = {
        'prob':     oof_prob,
        'pred_05':  oof_pred,
        'category': 'Model_A+SMOTE'
    }
    print(f"  {label:<20}  PR-AUC={pr:.3f}  F1={f1:.3f}  "
          f"Prec={prec:.3f}  Rec={rec:.3f}")

# ── Show which RF config wins ─────────────────────────────────────────────────
rf_candidates = {k: v for k, v in ALL_RESULTS.items()
                 if k.startswith('RF_SMOTE') or k == 'Model_A_RandomForest'}

best_rf_label = max(rf_candidates,
                    key=lambda k: average_precision_score(
                        y_all, rf_candidates[k]['prob']))
best_rf_pr    = average_precision_score(y_all, rf_candidates[best_rf_label]['prob'])

print(f"\n{'─'*55}")
print(f"  Plain RF PR-AUC  : "
      f"{average_precision_score(y_all, ALL_RESULTS['Model_A_RandomForest']['prob']):.3f}")
for s in RF_SMOTE_STRATS:
    lbl = f'RF_SMOTE_{s}'
    pr  = average_precision_score(y_all, ALL_RESULTS[lbl]['prob'])
    mk  = ' ← BEST RF CONFIG' if lbl == best_rf_label else ''
    print(f"  RF+SMOTE({s}) PR-AUC: {pr:.3f}{mk}")

if best_rf_label == 'Model_A_RandomForest':
    print(f"\n  ← BEST RF CONFIG: Plain RF{mk}")

print(f"\nBEST_RF_LABEL = '{best_rf_label}'  (PR-AUC={best_rf_pr:.3f})")
print("This will be auto-picked by Cell 11 for ensemble.")

RF + SMOTE variants  |  GroupKFold OOF
  RF_SMOTE_0.1          PR-AUC=0.685  F1=0.662  Prec=0.724  Rec=0.610
  RF_SMOTE_0.3          PR-AUC=0.676  F1=0.654  Prec=0.641  Rec=0.667
  RF_SMOTE_0.5          PR-AUC=0.677  F1=0.664  Prec=0.639  Rec=0.691
  RF_SMOTE_auto         PR-AUC=0.670  F1=0.667  Prec=0.637  Rec=0.699

───────────────────────────────────────────────────────
  Plain RF PR-AUC  : 0.687
  RF+SMOTE(0.1) PR-AUC: 0.685
  RF+SMOTE(0.3) PR-AUC: 0.676
  RF+SMOTE(0.5) PR-AUC: 0.677
  RF+SMOTE(auto) PR-AUC: 0.670

  ← BEST RF CONFIG: Plain RF

BEST_RF_LABEL = 'Model_A_RandomForest'  (PR-AUC=0.687)
This will be auto-picked by Cell 11 for ensemble.


In [ ]:
# ── Cell 8: TrAdaBoost — Fine-tune Model_I XGB booster on student data ─────────
# Continues training from booster_I (industrial XGBoost) on each student fold.
# Tests boost rounds: [20, 50, 100, 200, 500]
print("="*72)
print("TrAdaBoost: Transfer from Model_I booster  |  rounds [20,50,100,200,500]")
print("="*72)

TRADA_ROUNDS = [20, 50, 100, 200, 500]

xgb_t = {
    'objective': 'binary:logistic', 'eval_metric': 'logloss',
    'learning_rate': 0.05, 'max_depth': 4,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'seed': RANDOM_STATE, 'verbosity': 0,
}

oof_tra = {r: {'prob': np.zeros(n_all), 'pred': np.zeros(n_all, dtype=int)}
           for r in TRADA_ROUNDS}

hdr = f"{'Fold':>4}  {'TrPos':>5}  {'TePos':>5}  " + "  ".join(f"r{r:>3d}" for r in TRADA_ROUNDS)
print(f"\n{hdr}")
print("-"*(len(hdr)+2))

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_all, y_all, groups_all), 1):
    X_tr, X_va = X_all[tr_idx], X_all[va_idx]
    y_tr        = y_all[tr_idx]
    tr_pos      = int(y_tr.sum())
    va_pos      = int(y_all[va_idx].sum())
    neg_tr      = int((y_tr == 0).sum())
    spw         = round(neg_tr / max(tr_pos, 1), 2)

    if tr_pos < 2:
        for r in TRADA_ROUNDS:
            oof_tra[r]['prob'][va_idx] = oof_mi['XGBoost']['prob'][va_idx]
            oof_tra[r]['pred'][va_idx] = oof_mi['XGBoost']['pred'][va_idx]
        print(f"{fold:>4}  {tr_pos:>5}  {va_pos:>5}  " + "  ".join(["  skip"]*len(TRADA_ROUNDS)))
        continue

    Xti = ind_scaler.transform(X_tr)
    Xvi = ind_scaler.transform(X_va)
    dtr = xgb.DMatrix(Xti, label=y_tr, feature_names=FEATURE_COLS)
    dval= xgb.DMatrix(Xvi,             feature_names=FEATURE_COLS)
    xgb_t['scale_pos_weight'] = spw

    f1s = []
    for r in TRADA_ROUNDS:
        b = xgb.train(xgb_t, dtr, num_boost_round=r,
                      xgb_model=booster_I, verbose_eval=False)
        oof_tra[r]['prob'][va_idx] = b.predict(dval)
        oof_tra[r]['pred'][va_idx] = (oof_tra[r]['prob'][va_idx] >= 0.5).astype(int)
        f1s.append(f"{f1_score(y_all[va_idx], oof_tra[r]['pred'][va_idx], zero_division=0):>6.3f}")
    print(f"{fold:>4}  {tr_pos:>5}  {va_pos:>5}  " + "  ".join(f1s))

# ── Best TrAda by PR-AUC ──────────────────────────────────────────────────────
best_r = max(TRADA_ROUNDS, key=lambda r: average_precision_score(y_all, oof_tra[r]['prob']))
print(f"\n{'rounds':>8}  {'PR-AUC':>8}  {'F1@0.5':>8}")
for r in TRADA_ROUNDS:
    pr = average_precision_score(y_all, oof_tra[r]['prob'])
    f1 = f1_score(y_all, oof_tra[r]['pred'], zero_division=0)
    mk = " ← best PR-AUC" if r == best_r else ""
    print(f"  +{r:>4d}r  {pr:>8.3f}  {f1:>8.3f}{mk}")

# ── Save best TrAda model (full data) ─────────────────────────────────────────
X_all_ind  = ind_scaler.transform(X_all)
spw_f      = round((y_all==0).sum() / max(y_all.sum(),1), 2)
xgb_t['scale_pos_weight'] = spw_f
dtr_all    = xgb.DMatrix(X_all_ind, label=y_all, feature_names=FEATURE_COLS)
b_tra_fin  = xgb.train(xgb_t, dtr_all, num_boost_round=best_r,
                        xgb_model=booster_I, verbose_eval=False)
b_tra_fin.save_model(f"{PT_MODELS}/model_TrAda_data_class_+{best_r}rounds.json")

# ── Populate ALL_RESULTS ──────────────────────────────────────────────────────
for r in TRADA_ROUNDS:
    ALL_RESULTS[f'TrAda_+{r}r'] = {
        'prob': oof_tra[r]['prob'], 'pred_05': oof_tra[r]['pred'], 'category': 'TrAda'}

print(f"\nTrAdaBoost → ALL_RESULTS ({len(ALL_RESULTS)} entries)")
print(f"Best model (rounds={best_r}) saved → {PT_MODELS}")

TrAdaBoost: Transfer from Model_I booster  |  rounds [20,50,100,200,500]

Fold  TrPos  TePos  r 20  r 50  r100  r200  r500
--------------------------------------------------
   1    222     27   0.526   0.468   0.495   0.551   0.615
   2    191     58   0.525   0.549   0.552   0.592   0.600
   3    226     23   0.346   0.330   0.355   0.369   0.435
   4    188     61   0.563   0.592   0.583   0.611   0.662
   5    169     80   0.697   0.782   0.807   0.786   0.805

  rounds    PR-AUC    F1@0.5
  +  20r     0.561     0.555
  +  50r     0.647     0.569
  + 100r     0.656     0.584
  + 200r     0.673     0.606 ← best PR-AUC
  + 500r     0.664     0.643

TrAdaBoost → ALL_RESULTS (21 entries)
Best model (rounds=200) saved → /content/drive/MyDrive/CodeSmell_Stage3/data_class/trAda/models


In [ ]:
# ── Cell 9: TabPFN — Baseline (No SMOTE) ────────────────────────────────────
print("="*72)
print("TabPFN: Baseline — No SMOTE  |  GroupKFold OOF")
print("="*72)

from tabpfn import TabPFNClassifier

oof_tab_base = np.zeros(n_all)

print(f"\n{'Fold':>4}  {'TrPos':>5}  {'TePos':>5}  {'PR-AUC':>8}  {'F1@0.5':>8}")
print("-"*40)

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_all, y_all, groups_all), 1):
    X_tr, X_va = X_all[tr_idx], X_all[va_idx]
    y_tr, y_va  = y_all[tr_idx], y_all[va_idx]
    tr_pos      = int(y_tr.sum())
    if tr_pos < 2:
        print(f"{fold:>4}  {tr_pos:>5}  {int(y_va.sum()):>5}  {'skip':>8}")
        continue
    sc  = StandardScaler().fit(X_tr)
    mdl = TabPFNClassifier()
    mdl.fit(sc.transform(X_tr), y_tr)
    prob = mdl.predict_proba(sc.transform(X_va))[:, 1]
    oof_tab_base[va_idx] = prob
    pr = average_precision_score(y_va, prob)
    f1 = f1_score(y_va, (prob >= 0.5).astype(int), zero_division=0)
    print(f"{fold:>4}  {tr_pos:>5}  {int(y_va.sum()):>5}  {pr:>8.3f}  {f1:>8.3f}")

pr_all = average_precision_score(y_all, oof_tab_base)
f1_all = f1_score(y_all, (oof_tab_base>=0.5).astype(int), zero_division=0)
print(f"\nOVERALL  PR-AUC={pr_all:.3f}  F1@0.5={f1_all:.3f}")

ALL_RESULTS['TabPFN_NoSMOTE'] = {
    'prob':     oof_tab_base,
    'pred_05':  (oof_tab_base >= 0.5).astype(int),
    'category': 'TabPFN'
}
print(f"TabPFN baseline → ALL_RESULTS ({len(ALL_RESULTS)} entries)")

TabPFN: Baseline — No SMOTE  |  GroupKFold OOF

Fold  TrPos  TePos    PR-AUC    F1@0.5
----------------------------------------


tabpfn-v2.6-classifier-v2.6_default.ckpt:   0%|          | 0.00/43.0M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

   1    222     27     0.727     0.704
   2    191     58     0.676     0.604
   3    226     23     0.509     0.508
   4    188     61     0.774     0.757
   5    169     80     0.908     0.758

OVERALL  PR-AUC=0.694  F1@0.5=0.684
TabPFN baseline → ALL_RESULTS (22 entries)


In [ ]:
# ── Cell 10: TabPFN + SMOTE variants (0.1, 0.3, 0.5, 0.8, auto) + SMOTE-ENN ──
# Best configuration from summary: SMOTE(0.5) → PR-AUC 0.599
print("="*72)
print("TabPFN + SMOTE  |  strategies: 0.1 / 0.3 / 0.5 / 0.8 / auto")
print("TabPFN + SMOTE-ENN  |  strategies: 0.3 / 0.5 / auto")
print("="*72)

from tabpfn import TabPFNClassifier

def run_tabpfn_with_resampler(resampler_name, strategy, use_enn=False):
    oof = np.zeros(n_all)
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_all, y_all, groups_all), 1):
        X_tr, X_va = X_all[tr_idx], X_all[va_idx]
        y_tr, y_va  = y_all[tr_idx], y_all[va_idx]
        tr_pos      = int(y_tr.sum())
        if tr_pos < 2:
            continue
        sc   = StandardScaler().fit(X_tr)
        Xts  = sc.transform(X_tr)
        Xvs  = sc.transform(X_va)
        k    = min(3, tr_pos - 1) if tr_pos > 2 else 1
        smote_obj = SMOTE(sampling_strategy=strategy, k_neighbors=k, random_state=RANDOM_STATE)
        if use_enn:
            resampler = SMOTEENN(smote=smote_obj, random_state=RANDOM_STATE)
        else:
            resampler = smote_obj
        X_res, y_res = resampler.fit_resample(Xts, y_tr)
        mdl = TabPFNClassifier()
        mdl.fit(X_res, y_res)
        oof[va_idx] = mdl.predict_proba(Xvs)[:, 1]
    return oof

print(f"\n{'Configuration':<40}  {'PR-AUC':>7}  {'F1@0.5':>7}  {'Prec@0.5':>9}  {'Rec@0.5':>8}")
print("-"*75)

SMOTE_STRATEGIES = [0.1, 0.3, 0.5, 0.8, 'auto']

# SMOTE only
for strat in SMOTE_STRATEGIES:
    label = f"TabPFN_SMOTE_{strat}"
    prob  = run_tabpfn_with_resampler(f"SMOTE({strat})", strat, use_enn=False)
    pred  = (prob >= 0.5).astype(int)
    pr    = average_precision_score(y_all, prob)
    f1    = f1_score(y_all, pred, zero_division=0)
    prec  = precision_score(y_all, pred, zero_division=0)
    rec   = recall_score(y_all, pred, zero_division=0)
    ALL_RESULTS[label] = {'prob': prob, 'pred_05': pred, 'category': 'TabPFN+SMOTE'}
    mk = " ← best" if strat == 0.5 else ""
    print(f"  {label:<38}  {pr:>7.3f}  {f1:>7.3f}  {prec:>9.3f}  {rec:>8.3f}{mk}")

print()

# SMOTE-ENN (noise-cleaned)
for strat in [0.3, 0.5, 'auto']:
    label = f"TabPFN_SMOTEENN_{strat}"
    prob  = run_tabpfn_with_resampler(f"SMOTEENN({strat})", strat, use_enn=True)
    pred  = (prob >= 0.5).astype(int)
    pr    = average_precision_score(y_all, prob)
    f1    = f1_score(y_all, pred, zero_division=0)
    prec  = precision_score(y_all, pred, zero_division=0)
    rec   = recall_score(y_all, pred, zero_division=0)
    ALL_RESULTS[label] = {'prob': prob, 'pred_05': pred, 'category': 'TabPFN+SMOTE-ENN'}
    print(f"  {label:<38}  {pr:>7.3f}  {f1:>7.3f}  {prec:>9.3f}  {rec:>8.3f}")

print(f"\nTabPFN+SMOTE variants → ALL_RESULTS ({len(ALL_RESULTS)} entries)")

TabPFN + SMOTE  |  strategies: 0.1 / 0.3 / 0.5 / 0.8 / auto
TabPFN + SMOTE-ENN  |  strategies: 0.3 / 0.5 / auto

Configuration                              PR-AUC   F1@0.5   Prec@0.5   Rec@0.5
---------------------------------------------------------------------------
  TabPFN_SMOTE_0.1                          0.726    0.682      0.712     0.655
  TabPFN_SMOTE_0.3                          0.728    0.669      0.642     0.699
  TabPFN_SMOTE_0.5                          0.722    0.674      0.645     0.707 ← best
  TabPFN_SMOTE_0.8                          0.719    0.665      0.645     0.687
  TabPFN_SMOTE_auto                         0.707    0.652      0.635     0.671

  TabPFN_SMOTEENN_0.3                       0.693    0.637      0.594     0.687
  TabPFN_SMOTEENN_0.5                       0.686    0.643      0.567     0.743
  TabPFN_SMOTEENN_auto                      0.690    0.623      0.530     0.755

TabPFN+SMOTE variants → ALL_RESULTS (30 entries)


In [ ]:
# ── Cell 11: Ensemble — dynamic best components per smell ─────────────────────
print("="*72)
print("ENSEMBLE: Best RF + Best XGB + Best TabPFN  |  Stacking + Voting")
print("="*72)

from tabpfn import TabPFNClassifier

# ── AUTO-SELECT best RF and best TabPFN from ALL_RESULTS ──────────────────────
# RF: plain RF vs all RF+SMOTE variants
rf_candidates  = {k: v for k, v in ALL_RESULTS.items()
                  if k.startswith('RF_SMOTE') or k == 'Model_A_RandomForest'}
best_rf_key    = max(rf_candidates,
                     key=lambda k: average_precision_score(
                         y_all, rf_candidates[k]['prob']))

# TabPFN: any TabPFN+SMOTE variant
tab_candidates = {k: v for k, v in ALL_RESULTS.items()
                  if 'TabPFN' in k}
best_tab_key   = max(tab_candidates,
                     key=lambda k: average_precision_score(
                         y_all, tab_candidates[k]['prob']))

print(f"  RF  component  : {best_rf_key}  "
      f"(PR-AUC={average_precision_score(y_all, rf_candidates[best_rf_key]['prob']):.3f})")
print(f"  TabPFN component: {best_tab_key}  "
      f"(PR-AUC={average_precision_score(y_all, tab_candidates[best_tab_key]['prob']):.3f})")
print(f"  XGB component  : Model_A_XGBoost  "
      f"(PR-AUC={average_precision_score(y_all, ALL_RESULTS['Model_A_XGBoost']['prob']):.3f})")
print()

# ── Re-run OOF for the 3 chosen components ────────────────────────────────────
# (use stored OOF probs directly — no need to retrain)
oof_rf_e  = ALL_RESULTS[best_rf_key]['prob']
oof_xgb_e = ALL_RESULTS['Model_A_XGBoost']['prob']
oof_tab_e = ALL_RESULTS[best_tab_key]['prob']

# ── Stacking (LR meta-model) ──────────────────────────────────────────────────
X_meta      = np.column_stack([oof_rf_e, oof_xgb_e, oof_tab_e])
meta_model  = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
meta_model.fit(X_meta, y_all)
prob_stack  = meta_model.predict_proba(X_meta)[:, 1]
pred_stack  = (prob_stack >= 0.5).astype(int)

# ── Soft voting (simple average) ──────────────────────────────────────────────
prob_vote = (oof_rf_e + oof_xgb_e + oof_tab_e) / 3.0
pred_vote = (prob_vote >= 0.5).astype(int)

ens_label  = f'Ensemble_Stacking_{best_rf_key}_+_{best_tab_key}'
vote_label = f'Ensemble_Voting_{best_rf_key}_+_{best_tab_key}'

ALL_RESULTS[ens_label]  = {'prob': prob_stack, 'pred_05': pred_stack,
                            'category': 'Ensemble'}
ALL_RESULTS[vote_label] = {'prob': prob_vote,  'pred_05': pred_vote,
                            'category': 'Ensemble'}

for name, prob, pred in [(ens_label,  prob_stack, pred_stack),
                          (vote_label, prob_vote,  pred_vote)]:
    pr = average_precision_score(y_all, prob)
    f1 = f1_score(y_all, pred, zero_division=0)
    print(f"  {name}")
    print(f"    PR-AUC={pr:.3f}  F1={f1:.3f}")

print(f"\nEnsembles → ALL_RESULTS ({len(ALL_RESULTS)} entries)")

ENSEMBLE: Best RF + Best XGB + Best TabPFN  |  Stacking + Voting
  RF  component  : Model_A_RandomForest  (PR-AUC=0.687)
  TabPFN component: TabPFN_SMOTE_0.3  (PR-AUC=0.728)
  XGB component  : Model_A_XGBoost  (PR-AUC=0.647)

  Ensemble_Stacking_Model_A_RandomForest_+_TabPFN_SMOTE_0.3
    PR-AUC=0.718  F1=0.690
  Ensemble_Voting_Model_A_RandomForest_+_TabPFN_SMOTE_0.3
    PR-AUC=0.720  F1=0.694

Ensembles → ALL_RESULTS (32 entries)


In [ ]:
# ── Cell 12: Master Results Table ─────────────────────────────────────────────
# Columns:
#   Model | Category | PR-AUC | Opt_Thresh | F1@opt | Prec@opt | Rec@opt |
#   TP | FP | FN | F1@0.5 | Prec@0.5 | Rec@0.5 | Δ_vs_BestRule
# Sort: PR-AUC desc (primary), F1@opt desc (secondary)
# ★ BEST marks row #1
print("="*120)
print(" Data CLASS DETECTION — MASTER RESULTS TABLE")
print(f" Positives: {n_pos_all} / {n_all} ({n_pos_all/n_all*100:.1f}%)  |  GroupKFold(n={N_SPLITS})")
print("="*120)

def opt_threshold(y_true, y_prob):
    prec_c, rec_c, thr_c = precision_recall_curve(y_true, y_prob)
    f1_c   = 2 * prec_c * rec_c / (prec_c + rec_c + 1e-9)
    best_i = int(np.argmax(f1_c[:-1]))
    return float(thr_c[best_i])

def metrics_dual(y_true, prob, pred_05):
    pr_auc   = average_precision_score(y_true, prob)
    opt_thr  = opt_threshold(y_true, prob)
    pred_opt = (prob >= opt_thr).astype(int)

    # At optimal threshold
    f1_o   = f1_score(y_true, pred_opt, zero_division=0)
    pr_o   = precision_score(y_true, pred_opt, zero_division=0)
    re_o   = recall_score(y_true, pred_opt, zero_division=0)
    cm_o   = confusion_matrix(y_true, pred_opt)
    tp_o   = cm_o[1,1]; fp_o = cm_o[0,1]; fn_o = cm_o[1,0]

    # At 0.5
    f1_5   = f1_score(y_true, pred_05, zero_division=0)
    pr_5   = precision_score(y_true, pred_05, zero_division=0)
    re_5   = recall_score(y_true, pred_05, zero_division=0)

    return dict(pr_auc=round(pr_auc,3), opt_thr=round(opt_thr,3),
                f1_o=round(f1_o,3), pr_o=round(pr_o,3), re_o=round(re_o,3),
                tp=int(tp_o), fp=int(fp_o), fn=int(fn_o),
                f1_5=round(f1_5,3), pr_5=round(pr_5,3), re_5=round(re_5,3))

rows = []
for name, d in ALL_RESULTS.items():
    m = metrics_dual(y_all, d['prob'], d['pred_05'])
    rows.append({'Model': name, 'Category': d['category'], **m})

df_all = pd.DataFrame(rows)

# vs_BestRule baseline (PR-AUC delta)
best_rule_pr = df_all[df_all['Category']=='Rule']['pr_auc'].max()
df_all['delta_rule'] = (df_all['pr_auc'] - best_rule_pr).round(3)

# Sort: PR-AUC desc, F1@opt desc
df_all = df_all.sort_values(['pr_auc','f1_o'], ascending=[False,False]).reset_index(drop=True)

# Save
df_all.to_csv(f"{PA_RESULTS}/data_class_all_results.csv", index=False)

# ── Print ─────────────────────────────────────────────────────────────────────
CATEGORY_ORDER = ['Rule','Model_I','Model_A','TrAda','TabPFN',
                   'TabPFN+SMOTE','TabPFN+SMOTE-ENN','Ensemble']

W = 121
print(f"\n{'#':>2}  {'Model':<34} {'Category':>16}  "
      f"{'PR-AUC':>7}  {'OT':>6}  "
      f"{'F1@OT':>6} {'P@OT':>6} {'R@OT':>6}  "
      f"{'TP':>3} {'FP':>3} {'FN':>3}  "
      f"{'F1@.5':>6} {'P@.5':>5} {'R@.5':>5}  {'Δ_Rule':>7}")
print("─"*W)

prev_cat = None
for i, row in df_all.iterrows():
    cat = row['Category']
    if prev_cat is not None and cat != prev_cat:
        print("─"*W)
    prev_cat = cat
    star = " ★ BEST" if i == 0 else ""
    print(f"{i+1:>2}  {row['Model']:<34} {cat:>16}  "
          f"{row['pr_auc']:>7.3f}  {row['opt_thr']:>6.3f}  "
          f"{row['f1_o']:>6.3f} {row['pr_o']:>6.3f} {row['re_o']:>6.3f}  "
          f"{row['tp']:>3d} {row['fp']:>3d} {row['fn']:>3d}  "
          f"{row['f1_5']:>6.3f} {row['pr_5']:>5.3f} {row['re_5']:>5.3f}  "
          f"{row['delta_rule']:>+7.3f}{star}")

print("─"*W)
best = df_all.iloc[0]
print(f"\n🏆  WINNER : {best['Model']}")
print(f"    PR-AUC  : {best['pr_auc']:.3f}  (Δ vs best rule = {best['delta_rule']:+.3f})")
print(f"    Opt Thr : {best['opt_thr']:.3f}  →  F1={best['f1_o']:.3f}  "
      f"Prec={best['pr_o']:.3f}  Rec={best['re_o']:.3f}  TP={best['tp']}  FP={best['fp']}  FN={best['fn']}")
print(f"    @0.5    : F1={best['f1_5']:.3f}  Prec={best['pr_5']:.3f}  Rec={best['re_5']:.3f}")
print(f"\nTable saved → {PA_RESULTS}/data_class_all_results.csv")
# ── Save full results to CSV ───────────────────────────────────────────────────
csv_path = f"{PA_RESULTS}/{TARGET.replace('label_','')}_all_results.csv"
df_all.to_csv(csv_path, index=False)
print(f"\nFull results CSV saved → {csv_path}")
print(f"  {len(df_all)} models  |  columns: {list(df_all.columns)}")

 Data CLASS DETECTION — MASTER RESULTS TABLE
 Positives: 249 / 3266 (7.6%)  |  GroupKFold(n=5)

 #  Model                                      Category   PR-AUC      OT   F1@OT   P@OT   R@OT   TP  FP  FN   F1@.5  P@.5  R@.5   Δ_Rule
─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
 1  TabPFN_SMOTE_0.3                       TabPFN+SMOTE    0.728   0.645   0.695  0.725  0.667  166  63  83   0.669 0.642 0.699   +0.653 ★ BEST
 2  TabPFN_SMOTE_0.1                       TabPFN+SMOTE    0.726   0.354   0.698  0.658  0.743  185  96  64   0.682 0.712 0.655   +0.651
 3  TabPFN_SMOTE_0.5                       TabPFN+SMOTE    0.722   0.600   0.691  0.691  0.691  172  77  77   0.674 0.645 0.707   +0.647
─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
 4  Ensemble_Voting_Model_A_RandomForest_+_TabPFN_SMOTE_0.3         Ensemble    0.720   0.632   0.711  0.

In [ ]:
# ── ADD to Cell 13: Save RF+SMOTE final models ────────────────────────────────

# Find best RF config (same logic as Cell 11)
rf_candidates = {k: v for k, v in ALL_RESULTS.items()
                 if k.startswith('RF_SMOTE') or k == 'Model_A_RandomForest'}
best_rf_key   = max(rf_candidates,
                    key=lambda k: average_precision_score(
                        y_all, rf_candidates[k]['prob']))

print(f"Best RF config: {best_rf_key} — retraining on full data...")

# Retrain best RF+SMOTE on FULL student data
if 'SMOTE' in best_rf_key:
    strat = best_rf_key.split('_')[-1]
    strat = float(strat) if strat != 'auto' else 'auto'

    tr_pos = int(y_all.sum())
    tr_neg = int((y_all == 0).sum())
    if isinstance(strat, float):
        current_ratio = tr_pos / max(tr_neg, 1)
        safe_strat    = max(strat, current_ratio + 1e-6)
    else:
        safe_strat = strat

    k = min(3, tr_pos - 1) if tr_pos > 2 else 1
    X_res, y_res = SMOTE(
        sampling_strategy=safe_strat,
        k_neighbors=k,
        random_state=RANDOM_STATE
    ).fit_resample(X_all_sc, y_all)

    rf_best_final = RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=1,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    rf_best_final.fit(X_res, y_res)
    rf_path = f"{DEPLOY_DIR}/final_model_rf_smote_{strat}.joblib"
    joblib.dump(rf_best_final, rf_path)
    print(f"  RF+SMOTE({strat}) saved   → {rf_path}")

else:
    # plain RF is best — already saved earlier in Cell 13 as final_model_rf.joblib
    print(f"  Plain RF is best — already saved as final_model_rf.joblib")

# ── Save best TabPFN config ────────────────────────────────────────────────────
tab_candidates = {k: v for k, v in ALL_RESULTS.items() if 'TabPFN' in k}
best_tab_key   = max(tab_candidates,
                     key=lambda k: average_precision_score(
                         y_all, tab_candidates[k]['prob']))

print(f"\nBest TabPFN config: {best_tab_key} — retraining on full data...")

if 'SMOTE' in best_tab_key and 'ENN' not in best_tab_key:
    strat_tab = best_tab_key.split('_')[-1]
    strat_tab = float(strat_tab) if strat_tab != 'auto' else 'auto'
    resampler_type = 'SMOTE'
elif 'ENN' in best_tab_key:
    strat_tab = best_tab_key.split('_')[-1]
    strat_tab = float(strat_tab) if strat_tab != 'auto' else 'auto'
    resampler_type = 'SMOTEENN'
else:
    strat_tab      = None
    resampler_type = 'None'

if strat_tab is not None:
    tr_pos = int(y_all.sum())
    tr_neg = int((y_all == 0).sum())
    k      = min(3, tr_pos - 1) if tr_pos > 2 else 1
    if isinstance(strat_tab, float):
        safe_strat_tab = max(strat_tab, tr_pos / max(tr_neg, 1) + 1e-6)
    else:
        safe_strat_tab = strat_tab

    smote_obj = SMOTE(sampling_strategy=safe_strat_tab,
                      k_neighbors=k, random_state=RANDOM_STATE)
    if resampler_type == 'SMOTEENN':
        from imblearn.combine import SMOTEENN
        resampler = SMOTEENN(smote=smote_obj, random_state=RANDOM_STATE)
    else:
        resampler = smote_obj

    X_res_tab, y_res_tab = resampler.fit_resample(X_all_sc, y_all)
else:
    X_res_tab, y_res_tab = X_all_sc, y_all

from tabpfn import TabPFNClassifier
tab_final = TabPFNClassifier()
tab_final.fit(X_res_tab, y_res_tab)
tab_path  = f"{DEPLOY_DIR}/final_model_{best_tab_key}.pkl"
with open(tab_path, 'wb') as fh:
    pickle.dump(tab_final, fh)
print(f"  TabPFN model saved        → {tab_path}")

# ── Save ensemble meta-model ───────────────────────────────────────────────────
# Find the best ensemble from ALL_RESULTS
ens_candidates = {k: v for k, v in ALL_RESULTS.items()
                  if 'Ensemble' in k}
best_ens_key   = max(ens_candidates,
                     key=lambda k: average_precision_score(
                         y_all, ens_candidates[k]['prob']))
print(f"\nBest ensemble: {best_ens_key}")

# Rebuild and save meta-model if it's a stacking ensemble
if 'Stacking' in best_ens_key:
    oof_rf_e  = ALL_RESULTS[best_rf_key]['prob']
    oof_xgb_e = ALL_RESULTS['Model_A_XGBoost']['prob']
    oof_tab_e = ALL_RESULTS[best_tab_key]['prob']
    X_meta    = np.column_stack([oof_rf_e, oof_xgb_e, oof_tab_e])
    meta_final = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    meta_final.fit(X_meta, y_all)
    meta_path  = f"{DEPLOY_DIR}/final_ensemble_meta_lr.joblib"
    joblib.dump(meta_final, meta_path)
    print(f"  Stacking meta-LR saved    → {meta_path}")
else:
    print(f"  Voting ensemble — no meta-model to save (just average probabilities)")

# ── Update config JSON with best ensemble info ─────────────────────────────────
config['best_rf_config']    = best_rf_key
config['best_tabpfn_config']= best_tab_key
config['best_ensemble']     = best_ens_key
config['ensemble_components'] = {
    'rf':     best_rf_key,
    'xgb':    'Model_A_XGBoost',
    'tabpfn': best_tab_key,
}
Path(cfg_path).write_text(json.dumps(config, indent=2))
print(f"\n  Config JSON updated       → {cfg_path}")

# ── Final summary of ALL saved files ──────────────────────────────────────────
print(f"\n{'='*65}")
print("ALL DEPLOYMENT FILES:")
for f in sorted(os.listdir(DEPLOY_DIR)):
    size = os.path.getsize(f"{DEPLOY_DIR}/{f}") / 1024
    print(f"  {f:<55}  {size:>7.1f} KB")

Best RF config: Model_A_RandomForest — retraining on full data...
  Plain RF is best — already saved as final_model_rf.joblib

Best TabPFN config: TabPFN_SMOTE_0.3 — retraining on full data...


NameError: name 'X_all_sc' is not defined

In [ ]:
print('h')